# &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1387abff;">Chapter 2: </span>模型学习中的技巧

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;训练模型时的一个普遍现象是，前面层的参数变化会导致后面层的输入数据分布发生剧烈变化，这种不稳定性的传递使得训练过程变得困难，这被称为内部协变量偏移。我们多么想要一只强壮的大手，把数据的分布修改得健康一些啊！事实上，这种大手确实存在，请让我为你介绍Batch Normalization。

## &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #dab70aff;">2.3 BatchNorm与正则化</span>

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;简单来说，BatchNorm的核心思想是：​在网络的每一层输入之前，对数据进行标准化处理，使其均值为 0，方差为 1，强制每一层输入数据稳定。由于涉及反向传播与正向传播过程，BatchNorm需要作为一个单独的层工作，往往被放在在Affine层与激活函数层之间。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;BatchNorm的操作有两步：首先，计算数据均值与方差，然后根据结果对原数据进行缩放偏移，使得其变成均值为 0，方差为 1的新数据，这被称为标准化；其次，考虑到第一步可能使得输入数据丧失原有特征，我们引入超参数gamma与beta，gamma负责对新数据再次缩放，而beta负责对新数据再次偏移。请看我们的具体实践。

\begin{aligned}
\mu_B &= \frac{1}{m} \sum_{i=1}^{m} x_i \\
\sigma_B^2 &= \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2\\
\end{aligned}

\begin{aligned}
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}\\
y_i = \gamma \hat{x}_i + \beta
\end{aligned}

\begin{aligned}
y_i &= \gamma \frac{x_i - \mu_{\text{running}}}{\sqrt{\sigma_{\text{running}}^2 + \epsilon}} + \beta
\end{aligned}

\begin{aligned}
\mu_{\text{running}} &= \text{momentum} \cdot \mu_{\text{running}} + (1 - \text{momentum}) \cdot \mu_B \\
\sigma^2_{\text{running}} &= \text{momentum} \cdot \sigma^2_{\text{running}} + (1 - \text{momentum}) \cdot \sigma_B^2
\end{aligned}

In [1]:
import numpy as np

class BatchNormalization:
    def __init__(self,momentum):
        self.gamma = None
        self.beta = None
        self.momentum = momentum
        self.running_mean = 0
        self.running_var = 1
        self.x_hat = None
        self.dgamma = None
        self.dbeta = None
        self.means = None
        self.vars = None

        self.training = True

    def forward(self,x):
        if self.gamma is None:
            # 首次调用时初始化参数
            self.gamma = np.ones((1, x.shape[1]))
            self.beta = np.zeros((1, x.shape[1]))
            self.running_mean = np.zeros((1, x.shape[1]))
            self.running_var = np.ones((1, x.shape[1]))

        if self.training:
            self.means = np.mean(x, axis=0, keepdims=True)
            self.vars = np.var(x, axis=0, keepdims=True)
            
            self.running_mean = self.momentum * self.running_mean +\
                            (1 - self.momentum) * self.means
            self.running_var = self.momentum * self.running_var +\
                            (1 - self.momentum) * self.vars
            self.x_hat = (x - self.means) / np.sqrt(self.vars + 1e-7)
        else:
            self.x_hat = (x - self.running_mean) / np.sqrt(self.running_var + 1e-7)

        out = self.gamma * self.x_hat + self.beta

        return out
        
    def backward(self,dout):
        self.dbeta = np.sum(dout,axis = 0)
        self.dgamma = np.sum(dout * self.x_hat, axis = 0)
        S_dout = np.sum(dout, axis = 0)/ dout.shape[0]
        S_dout_xhat = np.sum(dout * self.x_hat, axis = 0)/ dout.shape[0]
        dx = (1 / np.sqrt(self.vars + 1e-7)) * \
                            (dout - S_dout - self.x_hat * S_dout_xhat) * self.gamma
        
        return dx
    
    def set_training(self, training):
        """设置训练/推理模式"""
        self.training = training


##### 本人编写此文档时发现Copilot居然能捕捉我的意图并且协助补全BatchNorm反向传播的部分，非常震惊。大模型的威力真是深不可测啊。
后注：作者编写本文档时LLM还远远没有读者你阅读时的那么智能，震惊是正常的。

\begin{aligned}
\frac{\partial L}{\partial \gamma} &= \sum_{i=1}^m \frac{\partial L}{\partial y_i} \cdot \hat{x}_i \\
\frac{\partial L}{\partial \beta} &= \sum_{i=1}^m \frac{\partial L}{\partial y_i} \\
\frac{\partial L}{\partial x_i} &= \frac{\gamma}{m\sqrt{\sigma^2 + \epsilon}} \left( m \cdot \frac{\partial L}{\partial y_i} - \sum_{j=1}^m \frac{\partial L}{\partial y_j} - \hat{x}_i \sum_{j=1}^m \frac{\partial L}{\partial y_j} \hat{x}_j \right)
\end{aligned}

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;前面的公式是向前传播的过程，第二部分公式则是反向传播的过程。笔者本来绘制了一张计算图，但是手写过于凌乱，故而没有放入文档中，有兴趣的读者自行前往文件夹寻找。实际上，BatchNorm的推理过程很特殊，为了追求模型行为一致性，我们不能使用推理时得到的数据平均数与方差完成BatchNorm，而是使用指数移动平均得到的平均数与方差，可以理解为我们实际上使用了历史数据的‘典型分布’进行推理。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;说了这么多BatchNorm的优点，我想，读者恐怕还是要见一见才真正意识得到吧！我们不妨就在这里搬出我们之前介绍的所有模型学习技巧，组装完成那个究极生物。你是否感觉到兴奋了呢？

In [2]:
import Chapter_the_first_FCNN as FCNN
import math
class Twolayersnet_BatchNorm:
    def __init__(self,input_size,hidden_size,output_size):
        self.params = {}#下面使用He初始化
        self.params['W1'] = np.random.randn(input_size,hidden_size)*math.sqrt(2/input_size)
        self.params['W2'] = np.random.randn(hidden_size,output_size)*math.sqrt(2/hidden_size)
        self.params['B1'] = np.zeros((1,hidden_size))
        self.params['B2'] = np.zeros((1,output_size))
        self.params['gamma1'] = np.ones((1,hidden_size))
        self.params['beta1'] = np.zeros((1,hidden_size))

        self.layers = FCNN.OrderedDict()
        self.layers['Affine1'] = FCNN.Affine(self.params['W1'],self.params['B1'])
        self.layers['BatchNorm1'] = BatchNormalization(momentum=0.9)
        self.layers['ReLU'] = FCNN.ReLU()
        self.layers['Affine2'] = FCNN.Affine(self.params['W2'],self.params['B2'])

        self.lastlayer = FCNN.SoftmaxWithLoss()
        
    def predict(self,x):
        for layer in self.layers.values():
            x= layer.forward(x)

        return x
        
    def loss(self,x,t):
        self.layers['BatchNorm1'].set_training(True)

        y = self.predict(x)
        return self.lastlayer.forward(y,t)

    def accuracy(self,x,t):
        self.layers['BatchNorm1'].set_training(False)
        y = self.predict(x)
        self.layers['BatchNorm1'].set_training(True)
        y = np.argmax(y,axis=1)
        if t.ndim !=1: t = np.argmax(t,axis=1)
        return np.sum(y==t)/float(x.shape[0])
        
    def gradient(self, x, t):
        self.loss(x, t)
        dout = 1
        dout = self.lastlayer.backward(dout)
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['W2'] = self.layers['Affine2'].dW
        grads['B1'] = self.layers['Affine1'].db
        grads['B2'] = self.layers['Affine2'].db
        grads['gamma1'] = self.layers['BatchNorm1'].dgamma
        grads['beta1'] = self.layers['BatchNorm1'].dbeta

        return grads

In [3]:
import Chapter_the_second_1_op as OPTIMIZER
if __name__ == '__main__':
    (x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = FCNN.mnist.load_data()
    x_train = x_train_raw.reshape(x_train_raw.shape[0], -1)/255.0
    x_test = x_test_raw.reshape(x_test_raw.shape[0], -1)/255.0
    y_train = FCNN.to_one_hot(y_train_raw)
    y_test = FCNN.to_one_hot(y_test_raw)
    print(f"\n预处理后:")
    print(f"x_train range: [{x_train.min():.3f}, {x_train.max():.3f}]")
    print(f"x_test range: [{x_test.min():.3f}, {x_test.max():.3f}]")

    network = Twolayersnet_BatchNorm(input_size=784, hidden_size=50, output_size=10)

    iters = 10000
    train_size = x_train.shape[0]
    batch_size = 100 
    train_loss_list = []  
    train_acc_list = []
    test_acc_list = []  

    iter_per_epoch = max(train_size / batch_size, 1)  
    optimizer = OPTIMIZER.Adam(lr=0.001,beta1=0.9,beta2=0.999)  # 先创建优化器实例

    for i in range(iters):
        batch_mask = np.random.choice(train_size, batch_size)
        x_batch = x_train[batch_mask]
        y_batch = y_train[batch_mask]
        
        network.layers['BatchNorm1'].set_training(True)
        grad = network.gradient(x_batch, y_batch)
        
        optimizer.update(network.params, grad)

        loss = network.loss(x_batch, y_batch)
        train_loss_list.append(loss)

        if i % iter_per_epoch == 0:
            train_acc = network.accuracy(x_train, y_train)
            test_acc = network.accuracy(x_test, y_test)
            train_acc_list.append(train_acc)
            test_acc_list.append(test_acc)
            print("training accuracy:",train_acc, 'testing accuracy:',test_acc, 'training loss:',train_loss_list[-1])
            # 在前向传播中添加调试输出
            print("W1 mean:", np.mean(network.params['W1']), "grad mean:", np.mean(grad['W1']))


预处理后:
x_train range: [0.000, 1.000]
x_test range: [0.000, 1.000]
training accuracy: 0.1572 testing accuracy: 0.16 training loss: 2.227494853642383
W1 mean: -0.0004287064996386295 grad mean: 0.00084101009466391
training accuracy: 0.9389333333333333 testing accuracy: 0.9378 training loss: 0.18908047452364918
W1 mean: -0.0029368725653940173 grad mean: -5.336442482728519e-05
training accuracy: 0.9563166666666667 testing accuracy: 0.9529 training loss: 0.21866916134210596
W1 mean: -0.003591894573916168 grad mean: -5.1489373458223216e-05
training accuracy: 0.96535 testing accuracy: 0.9581 training loss: 0.14155999683796416
W1 mean: -0.0037362359959169466 grad mean: -1.3488315526669866e-05
training accuracy: 0.9725666666666667 testing accuracy: 0.9635 training loss: 0.11729371495342647
W1 mean: -0.0035243748069917003 grad mean: -6.91290028922425e-05
training accuracy: 0.9753333333333334 testing accuracy: 0.9652 training loss: 0.15958493830040058
W1 mean: -0.003709353838658475 grad mean: 9.41

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;我们的正确率大概只有97%，似乎哪里不对？你现在的失落感就好像孙悟空西天取经，发现佛祖真的就只给了经书，打开看还是没有字的。忙里忙外，我们距离幻想中的百分之百正确率还有很远。明明我们已经竭尽所能，这到底是怎么回事？

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;你可能嗅察到了一些蛛丝马迹：训练正确率高于测试正确率约1%。这说明模型在训练时表现还不错，一上真刀枪就有些软弱。或者说，模型对于训练数据太过熟悉，导致对陌生的测试数据有抵触感，这被称为‘过拟合’。为了解决这种问题，我们需要正则化。

In [4]:
class Dropout:
    def __init__(self, dropout_ratio):
        self.dropout_ratio = dropout_ratio
        self.mask = None
        self.training = True  # 添加训练模式标志

    def forward(self, x, training = None):
        self.training = training
        if training:
            self.mask = np.random.rand(*x.shape) > self.dropout_ratio
            return x * self.mask/ (1.0 - self.dropout_ratio)
        else:
            return x 

    def backward(self, dout):
        if self.training and self.dropout_ratio < 1.0:
            return dout * self.mask / (1.0 - self.dropout_ratio)
        return dout
    
    def set_training(self, training):
        """设置训练/推理模式"""
        self.training = training

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;以上算法被称为Dropout。核心思想是，为了防止对训练数据集过拟合，我们在训练时随机丢弃一些神经元。这样迫使每个神经元不可过度依赖其他神经元，防止网络对于训练数据的噪声过于敏感，提升了模型泛化能力。不过由于我们丢弃了一些输出，为了保持期望的一致，我们对留下的输出进行放大。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Dropout是强大的工具，然而往往与BatchNorm略冲突。原因是前者破坏数据的分布，BatchNorm却又将分布标准化。所以我们一般把Dropout置于BatchNorm之后。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;另一种正则化方法是权重衰减。我们试图抑制单个神经元过于强大的发生，因此我们对参数的变大设置惩罚。这种惩罚一般直接写入损失函数，分为L1正则化与L2正则化，我们重点介绍后者。

$$
J(\theta) = J_0(\theta) + \frac{\lambda}{2} \left( \sum w_1^2 + \sum w_2^2 + \cdots + \sum w_L^2 \right)
$$

$$
\frac{\partial J}{\partial w} = \frac{\partial J_0}{\partial w} + \lambda w
$$

$$
w^{(t+1)} = (1 - \eta\lambda)w^{(t)} - \eta \frac{\partial J_0}{\partial w}
$$

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;在L2正则化中，我们以参数平方和形式追加惩罚，参数梯度也要随之变化。至于在神经网络中应用，一种方法是则直接在原神经网络上修改，另一种是包装成一个函数。我们推荐后者。

In [5]:
def l2_regularized_loss(network, x, t, L2_lambda=0.01):
    """
    L2正则化损失计算包装器
    :param network: 神经网络实例
    :param x: 输入数据
    :param t: 目标标签
    :param l2_lambda: L2正则化系数
    :return: 带L2正则化的总损失
    """
    # 计算原始损失
    original_loss = network.loss(x, t)
    
    # 计算L2正则化项
    l2_penalty = 0
    for key in network.params.keys():
        if key.startswith('W'):  # 只对权重矩阵应用L2
            l2_penalty += 0.5 * L2_lambda * np.sum(network.params[key]**2)
    
    return original_loss + l2_penalty

def l2_regularized_gradient(network, x, t, L2_lambda=0.01):
    
    # 计算原始梯度
    grad = network.gradient(x, t)
    
    # 添加L2正则化梯度
    for key in grad.keys():
        if key.startswith('W'):  # 只对权重矩阵应用L2
            grad[key] += L2_lambda * network.params[key]
    
    return grad

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #e70e36ff;">接下来是我们的全力一击！</span>看看我们一路走来都干了什么吧：全连接神经网络，激活函数，Adam，He初始化，BatchNorm，Dropout，L2权重衰减。最后的最后，为了加速我们的训练过程，我们加入早停策略与学习率衰减。策略很简单，如果连续多次模型性能不提升，我们直接停止训练。而为了保证后期参数更新不振荡，我们逐步减小学习率。请看看吧，下面是第二章的最后内容。

In [6]:
class Threelayersnet_BatchNorm_Dropout:
    def __init__(self,input_size,hidden_size1,hidden_size2,output_size):
        self.params = {}#下面使用He初始化
        self.params['W1'] = np.random.randn(input_size,hidden_size1)*math.sqrt(2/input_size)
        self.params['W2'] = np.random.randn(hidden_size1,hidden_size2)*math.sqrt(2/hidden_size1)
        self.params['W3'] = np.random.randn(hidden_size2,output_size)*math.sqrt(2/hidden_size2)

        self.params['B1'] = np.zeros((1,hidden_size1))
        self.params['B2'] = np.zeros((1,hidden_size2))
        self.params['B3'] = np.zeros((1,output_size))

        self.params['gamma1'] = np.ones((1,hidden_size1))
        self.params['beta1'] = np.zeros((1,hidden_size1))
        self.params['gamma2'] = np.ones((1,hidden_size2))
        self.params['beta2'] = np.zeros((1,hidden_size2))

        self.layers = FCNN.OrderedDict()
        self.layers['Affine1'] = FCNN.Affine(self.params['W1'],self.params['B1'])
        self.layers['BatchNorm1'] = BatchNormalization(momentum=0.9)
        self.layers['ReLU1'] = FCNN.ReLU()
        self.layers['Affine2'] = FCNN.Affine(self.params['W2'],self.params['B2'])
        self.layers['BatchNorm2'] = BatchNormalization(momentum=0.9)
        self.layers['ReLU2'] = FCNN.ReLU()
        self.layers['Dropout1'] = Dropout(dropout_ratio=0.3)
        self.layers['Affine3'] = FCNN.Affine(self.params['W3'],self.params['B3'])

        self.lastlayer = FCNN.SoftmaxWithLoss()
        
    def predict(self,x):
        for layer in self.layers.values():
            x= layer.forward(x)

        return x
        
    def loss(self,x,t):
        for layer_name, layer in self.layers.items():
            if hasattr(layer, 'set_training'):
                layer.set_training(True)

        y = self.predict(x)
        return self.lastlayer.forward(y,t)

    def accuracy(self,x,t):
        for layer_name, layer in self.layers.items():
            if hasattr(layer, 'set_training'):
                layer.set_training(False)
        y = self.predict(x)
        # 恢复为训练模式
        for layer_name, layer in self.layers.items():
            if hasattr(layer, 'set_training'):
                layer.set_training(True)
        y = np.argmax(y,axis=1)
        if t.ndim !=1: t = np.argmax(t,axis=1)
        return np.sum(y==t)/float(x.shape[0])
        
    def gradient(self, x, t):
        self.loss(x, t)
        dout = 1
        dout = self.lastlayer.backward(dout)
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['W2'] = self.layers['Affine2'].dW
        grads['W3'] = self.layers['Affine3'].dW
        grads['B1'] = self.layers['Affine1'].db
        grads['B2'] = self.layers['Affine2'].db
        grads['B3'] = self.layers['Affine3'].db
        grads['gamma1'] = self.layers['BatchNorm1'].dgamma
        grads['beta1'] = self.layers['BatchNorm1'].dbeta
        grads['gamma2'] = self.layers['BatchNorm2'].dgamma
        grads['beta2'] = self.layers['BatchNorm2'].dbeta

        return grads

### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;这是神圣的时刻，请屏住呼吸，运行下面这个单元格，蒙上眼睛等待约十分钟。
#### <span style="color: #08af85ff;">真的...真的...绕了好大一圈啊...</span>

In [7]:
if __name__ == '__main__':
    network = Threelayersnet_BatchNorm_Dropout(input_size=784, hidden_size1=512, hidden_size2=256, output_size=10)
    iters = 10000
    train_size = x_train.shape[0]
    batch_size = 256 
    train_loss_list = []  
    train_acc_list = []
    test_acc_list = []  
    best_test_acc = 0
    patience = 10  # 早停耐心值
    no_improve_count = 0

    iter_per_epoch = int(np.ceil(train_size / batch_size)) 
    optimizer = OPTIMIZER.Adam(lr=0.005, beta1=0.9, beta2=0.999)

    for i in range(iters):
        batch_mask = np.random.choice(train_size, batch_size)
        x_batch = x_train[batch_mask]
        y_batch = y_train[batch_mask]
        
        for layer_name, layer in network.layers.items():
            if hasattr(layer, 'set_training'):
                layer.set_training(True)

        loss = l2_regularized_loss(network, x_batch, y_batch, L2_lambda=0.00002)
        grad = l2_regularized_gradient(network, x_batch, y_batch, L2_lambda=0.00002)
        '''grad = network.gradient(x_batch, y_batch)
        loss = network.loss(x_batch, y_batch)'''
        
        optimizer.update(network.params, grad)

        train_loss_list.append(loss)
        # 在训练循环中添加学习率衰减
        if i % 2000 == 0 and i > 0:
            optimizer.lr *= 0.95  

        if i % iter_per_epoch == 0:
            train_acc = network.accuracy(x_train, y_train)
            test_acc = network.accuracy(x_test, y_test)
            train_acc_list.append(train_acc)
            test_acc_list.append(test_acc)

            if test_acc > best_test_acc:
                best_test_acc = test_acc
                no_improve_count = 0
            else:
                no_improve_count += 1
                
            if no_improve_count >= patience:
                print(f"Early stopping at iteration {i}")
                print(f"Best test accuracy: {best_test_acc:.4f}")
                break
                
            print(f'iter: {i}, train_acc: {train_acc:.4f}, test_acc: {test_acc:.4f}, loss: {loss:.4f}')
            print(f"Learning rate: {optimizer.lr:.6f},no_improve_count: {no_improve_count}")
            # 在前向传播中添加调试输出
            print("W1 mean:", np.mean(network.params['W1']), "grad mean:", np.mean(grad['W1']))

iter: 0, train_acc: 0.6825, test_acc: 0.6940, loss: 2.7640
Learning rate: 0.005000,no_improve_count: 0
W1 mean: -2.7672479728711148e-05 grad mean: 7.5261197550422555e-06
iter: 235, train_acc: 0.9750, test_acc: 0.9672, loss: 0.1484
Learning rate: 0.005000,no_improve_count: 0
W1 mean: -0.0022812727737860274 grad mean: -1.9458438996412283e-05
iter: 470, train_acc: 0.9841, test_acc: 0.9756, loss: 0.0971
Learning rate: 0.005000,no_improve_count: 0
W1 mean: -0.0028487404636584634 grad mean: 9.101650582420774e-07
iter: 705, train_acc: 0.9837, test_acc: 0.9724, loss: 0.0612
Learning rate: 0.005000,no_improve_count: 1
W1 mean: -0.0030092628657627848 grad mean: 3.1714351723891286e-06
iter: 940, train_acc: 0.9880, test_acc: 0.9752, loss: 0.0717
Learning rate: 0.005000,no_improve_count: 2
W1 mean: -0.0029260807611431092 grad mean: -4.213394589180827e-06
iter: 1175, train_acc: 0.9871, test_acc: 0.9761, loss: 0.0554
Learning rate: 0.005000,no_improve_count: 0
W1 mean: -0.0026992138976558427 grad mea

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;哈哈，你对这个结果还满意吗？不要露出那样对笔者不信任的表情啊！！什么叫做训练了半天还只有98%，已经很厉害了好吗？不过，全连接神经网络是有极限的，越是工于心计地优化，越是容易在意想不到的地方跌倒。所以，我们不做全连接神经网络啦！请看下一章，我们将为你介绍更高级的卷积神经网络 ：）。